# Eksperimen Preprocessing — Credit Card Fraud Dataset
**Nama:** Yogi-Dharma  
**Dataset:** Credit Card Fraud Detection  
**Deskripsi:** Dataset ini berisi 100.000 transaksi kartu kredit dengan label biner `IsFraud` (1 = fraud, 0 = tidak fraud). Tujuan preprocessing ini adalah mempersiapkan data agar siap dilatih oleh model machine learning.

---

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded successfully!')

---
## 2. Data Loading

In [ ]:
# Load dataset dari folder raw
df = pd.read_csv('../credit_card_fraud_dataset_raw.csv')
print(f'Dataset berhasil dimuat!')
print(f'Jumlah baris   : {df.shape[0]:,}')
print(f'Jumlah kolom   : {df.shape[1]}')
print(f'Kolom          : {df.columns.tolist()}')

In [ ]:
# Tampilkan 5 baris pertama
df.head()

In [ ]:
# Informasi tipe data
df.info()

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Statistik Deskriptif

In [ ]:
df.describe()

### 3.2 Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)
print(f'\nTotal missing values: {missing.sum()}')

### 3.3 Distribusi Target Variable (IsFraud)

In [ ]:
fraud_counts = df['IsFraud'].value_counts()
fraud_pct = df['IsFraud'].value_counts(normalize=True) * 100

print('Distribusi IsFraud:')
print(f'  Tidak Fraud (0): {fraud_counts[0]:,} ({fraud_pct[0]:.2f}%)')
print(f'  Fraud       (1): {fraud_counts[1]:,} ({fraud_pct[1]:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Tidak Fraud (0)', 'Fraud (1)'], fraud_counts.values,
            color=['steelblue', 'tomato'], edgecolor='white', linewidth=0.8)
axes[0].set_title('Distribusi Kelas IsFraud', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Jumlah Transaksi')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(fraud_counts.values, labels=['Tidak Fraud (0)', 'Fraud (1)'],
            autopct='%1.2f%%', colors=['steelblue', 'tomato'],
            startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Proporsi Kelas IsFraud', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n[NOTE] Dataset sangat imbalanced: hanya 1% transaksi fraud!')

### 3.4 Distribusi Amount

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram Amount
axes[0].hist(df['Amount'], bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribusi Amount', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frekuensi')

# Boxplot Amount by IsFraud
df.boxplot(column='Amount', by='IsFraud', ax=axes[1],
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='tomato', linewidth=2))
axes[1].set_title('Amount berdasarkan IsFraud', fontsize=13, fontweight='bold')
axes[1].set_xlabel('IsFraud')
axes[1].set_ylabel('Amount')
plt.suptitle('')

plt.tight_layout()
plt.show()

### 3.5 Distribusi Kolom Kategorikal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# TransactionType
tt_counts = df['TransactionType'].value_counts()
axes[0].bar(tt_counts.index, tt_counts.values, color=['steelblue', 'coral'], edgecolor='white')
axes[0].set_title('Distribusi TransactionType', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Jumlah')
for i, v in enumerate(tt_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

# Location
loc_counts = df['Location'].value_counts()
axes[1].barh(loc_counts.index, loc_counts.values, color='steelblue', edgecolor='white')
axes[1].set_title('Distribusi Location', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Jumlah')

plt.tight_layout()
plt.show()

### 3.6 Correlation Heatmap (Fitur Numerik)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr()

plt.figure(figsize=(8, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, square=True, vmin=-1, vmax=1)
plt.title('Correlation Matrix — Fitur Numerik', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Preprocessing

In [ ]:
# Salin dataframe agar data asli tetap terjaga
df_prep = df.copy()
print('DataFrame disalin. Shape:', df_prep.shape)

### 4.1 Drop Kolom Tidak Diperlukan
`TransactionID` hanya berperan sebagai identifier unik dan tidak memiliki nilai prediktif untuk model.

In [ ]:
df_prep = df_prep.drop(columns=['TransactionID'])
print('Kolom TransactionID dihapus.')
print('Kolom tersisa:', df_prep.columns.tolist())

### 4.2 Handle Missing Values
Berdasarkan EDA, dataset ini tidak memiliki missing values. Namun kita tetap menjalankan penanganan agar pipeline bersifat robust.

In [ ]:
missing_total = df_prep.isnull().sum().sum()
print(f'Total missing values: {missing_total}')

if missing_total > 0:
    for col in df_prep.columns:
        if df_prep[col].isnull().sum() > 0:
            if df_prep[col].dtype in [np.float64, np.int64]:
                df_prep[col].fillna(df_prep[col].median(), inplace=True)
                print(f'  {col}: diisi median')
            else:
                df_prep[col].fillna(df_prep[col].mode()[0], inplace=True)
                print(f'  {col}: diisi modus')
else:
    print('Tidak ada missing values — tidak ada tindakan yang diperlukan.')

### 4.3 Ekstraksi Fitur dari TransactionDate
Kolom `TransactionDate` mengandung informasi waktu yang berguna jika diekstrak menjadi fitur terpisah:
- `transaction_hour` — jam transaksi terjadi
- `transaction_day` — hari dalam bulan
- `transaction_month` — bulan transaksi
- `transaction_dayofweek` — hari dalam minggu (0=Senin)

In [ ]:
df_prep['TransactionDate'] = pd.to_datetime(df_prep['TransactionDate'])

df_prep['transaction_hour']      = df_prep['TransactionDate'].dt.hour
df_prep['transaction_day']       = df_prep['TransactionDate'].dt.day
df_prep['transaction_month']     = df_prep['TransactionDate'].dt.month
df_prep['transaction_dayofweek'] = df_prep['TransactionDate'].dt.dayofweek

df_prep = df_prep.drop(columns=['TransactionDate'])

print('Fitur datetime berhasil diekstrak:')
print(df_prep[['transaction_hour','transaction_day','transaction_month','transaction_dayofweek']].head())

In [ ]:
# Visualisasi distribusi jam transaksi fraud vs tidak fraud
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, fraud_val, label, color in zip(
    axes, [0, 1], ['Tidak Fraud', 'Fraud'], ['steelblue', 'tomato']
):
    subset = df_prep[df_prep['IsFraud'] == fraud_val]['transaction_hour']
    ax.hist(subset, bins=24, range=(0, 24), color=color, edgecolor='white', linewidth=0.5)
    ax.set_title(f'Distribusi Jam Transaksi — {label}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Jam')
    ax.set_ylabel('Jumlah')

plt.tight_layout()
plt.show()

### 4.4 Encoding Kolom Kategorikal

In [ ]:
# --- TransactionType: Binary Encoding ---
# purchase = 1, refund = 0
df_prep['TransactionType'] = df_prep['TransactionType'].map({'purchase': 1, 'refund': 0})
print('TransactionType setelah encoding:')
print(df_prep['TransactionType'].value_counts())

In [ ]:
# --- Location: Label Encoding ---
le = LabelEncoder()
df_prep['Location'] = le.fit_transform(df_prep['Location'])
print('Location setelah Label Encoding:')
print('Mapping:', dict(zip(le.classes_, le.transform(le.classes_))))
print(df_prep['Location'].value_counts())

### 4.5 Feature Scaling (StandardScaler)
StandardScaler diterapkan pada semua fitur kecuali target `IsFraud`, agar model tidak bias terhadap fitur dengan skala besar.

In [ ]:
feature_cols = [c for c in df_prep.columns if c != 'IsFraud']

scaler = StandardScaler()
df_prep[feature_cols] = scaler.fit_transform(df_prep[feature_cols])

print('StandardScaler diterapkan pada:', feature_cols)
print()
df_prep[feature_cols].describe().round(4)

---
## 5. Hasil Akhir Preprocessing

In [ ]:
print('Shape akhir:', df_prep.shape)
print('Kolom:', df_prep.columns.tolist())
print()
df_prep.head()

In [ ]:
# Verifikasi tidak ada missing values di akhir
print('Missing values di data hasil preprocessing:')
print(df_prep.isnull().sum())

In [ ]:
# Simpan hasil preprocessing
output_dir = 'credit_card_fraud_dataset_preprocessing'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'credit_card_fraud_preprocessing.csv')

df_prep.to_csv(output_path, index=False)
print(f'Dataset preprocessing tersimpan di: {output_path}')
print(f'Shape: {df_prep.shape}')

---
## 6. Ringkasan Preprocessing

| Tahap | Tindakan | Keterangan |
|---|---|---|
| Drop kolom | Hapus `TransactionID` | Hanya identifier, tidak prediktif |
| Missing values | Tidak ada tindakan | Tidak ditemukan missing values |
| Datetime | Ekstrak `hour`, `day`, `month`, `dayofweek` | Dari `TransactionDate` |
| Encoding | `TransactionType` → binary (purchase=1, refund=0) | Hanya 2 nilai unik |
| Encoding | `Location` → Label Encoding (0–9) | 10 kota unik |
| Scaling | StandardScaler pada semua fitur kecuali `IsFraud` | Agar skala seragam |

**Dataset siap dilatih dengan shape:** `(100000, 9)`  
**Fitur:** `Amount`, `MerchantID`, `TransactionType`, `Location`, `transaction_hour`, `transaction_day`, `transaction_month`, `transaction_dayofweek`  
**Target:** `IsFraud`